# Single carrier

> One carrier: every symbol of a transmission sees the same channel coefficient.

In [ ]:
#| default_exp waveform.single_carrier

In [ ]:
#| hide
from nbdev.showdoc import *

A waveform lays a transmission's symbols onto time and frequency, which sets the channel each one sees. It is a
stage of a link chain (see `radio.link.chain`): from the fading's `h` at each subcarrier, `(B, K)`, and the transmit
power, it gives `h_sym`, the channel of each symbol, `√P` included, broadcast against the symbols `x`.

The waveform is not the multiple access: how users share the medium (TDMA, ALOHA) is the protocol's (see
`net.protocol`), and a TDMA slot can carry either waveform.

`SingleCarrier` is one carrier: every symbol sees one coefficient, `h` if the fading is flat, else
`√(mean |h|²)` over the subcarriers (the same power, see `effective_coefficient`). `OFDM` (see `waveform.ofdm`)
gives each symbol its subcarrier's `h` instead.

In [ ]:
#| export
from __future__ import annotations

import numpy as np

from comm_core.block import Block, _is_torch
from comm_core.channel.fading import effective_coefficient

In [ ]:
#| export
def _like(values, x):
    "numpy values as `x`'s kind: a tensor of its complex type on its device, or numpy."
    if _is_torch(x):
        import torch
        return torch.as_tensor(values, dtype=x.dtype, device=x.device)
    return values


class SingleCarrier(Block):
    "One carrier: every symbol of a transmission sees one coefficient (√(mean |h|²) over the subcarriers), √P included."

    outputs = ('h_sym',)

    def call(
        self,
        x, # The symbols, (B, ..., S)
        h, # h of each transmission at each subcarrier, (B, K)
        power, # Their transmit powers, (B,)
    ):
        "The channel of each symbol, √P included, (B, 1, ..., 1)."
        c = np.array([effective_coefficient(h[b]) * float(power[b]) ** 0.5 for b in range(len(h))])
        return _like(c.reshape((len(c),) + (1,) * (x.ndim - 1)), x)

In [ ]:
h_sym = SingleCarrier()(np.zeros((2, 3, 8)), np.array([[1j], [2.0]]), np.array([4.0, 1.0]))
assert h_sym.shape == (2, 1, 1) and h_sym[:, 0, 0].tolist() == [2j, 2.0]
assert abs(SingleCarrier()(np.zeros((1, 4)), np.array([[3.0, 4j]]), np.ones(1))[0, 0] - np.sqrt(12.5)) < 1e-12

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()